# ПЗ 8 — Постобработка результатов

**Задача:** устранить косяки после распознавания:
- **Для текста (OCR):** дедупликация похожих строк (Levenshtein/fuzzy matching)
- **Для детекций (YOLO/ResNet/LLM):** склейка сработавших детекций по временны́м окнам (temporal merging)

**Стек:** `pandas`, `rapidfuzz`, `scipy`, `numpy`

In [ ]:
!pip install rapidfuzz pandas scipy numpy -q

## Часть 1 — Дедупликация текста (OCR-субтитры)

In [ ]:
import pandas as pd
from rapidfuzz import fuzz, process

OCR_CSV = '/content/outputs/ocr_results/subtitles.csv'
df_ocr = pd.read_csv(OCR_CSV)
print(f'Строк до дедупликации: {len(df_ocr)}')

def deduplicate_texts(texts: list, threshold: int = 85) -> list:
    """Убрать строки, похожие на уже виденные (fuzzy ratio > threshold)."""
    unique = []
    for t in texts:
        if not unique:
            unique.append(t)
            continue
        best_score = process.extractOne(t, unique, scorer=fuzz.ratio)
        if best_score is None or best_score[1] < threshold:
            unique.append(t)
    return unique

unique_texts = deduplicate_texts(df_ocr['text'].dropna().tolist())
print(f'Строк после дедупликации (порог 85%): {len(unique_texts)}')
df_dedup = pd.DataFrame({'text_dedup': unique_texts})
df_dedup.to_csv('/content/outputs/ocr_results/subtitles_dedup.csv', index=False)
df_dedup.head(20)

## Часть 2 — Склейка детекций по временным окнам (YOLO)

In [ ]:
import numpy as np

DET_CSV = '/content/outputs/detections/yolo_detections.csv'
df_det = pd.read_csv(DET_CSV)

# Извлечь номер кадра из имени файла
df_det['frame_num'] = df_det['frame'].str.extract(r'(\d+)').astype(int)

# Склейка: объект считается «одним событием» если появляется
# в соседних кадрах (gap <= WINDOW) с confidence выше порога
WINDOW = 5  # кадров

def merge_detections(group):
    group = group.sort_values('frame_num').reset_index(drop=True)
    events = []
    start = group.iloc[0]
    prev_frame = start['frame_num']
    for _, row in group.iloc[1:].iterrows():
        if row['frame_num'] - prev_frame > WINDOW:
            events.append({'class_name': start['class_name'],
                           'start_frame': start['frame_num'],
                           'end_frame':   prev_frame,
                           'avg_conf':    round(group['confidence'].mean(), 3)})
            start = row
        prev_frame = row['frame_num']
    events.append({'class_name': start['class_name'],
                   'start_frame': start['frame_num'],
                   'end_frame':   prev_frame,
                   'avg_conf':    round(group['confidence'].mean(), 3)})
    return pd.DataFrame(events)

merged = df_det.groupby('class_name', group_keys=False).apply(merge_detections)
merged = merged.sort_values('start_frame').reset_index(drop=True)
merged.to_csv('/content/outputs/detections/yolo_merged.csv', index=False)
print(f'Событий после склейки: {len(merged)}')
print(merged.to_string())